In [ ]:
%pip install prophet pandas
from prophet import Prophet
import pandas as pd
def train_prophet(df):
    data = df.rename(columns={
        "Date": "ds",
        "Close": "y"
    })
    model = Prophet()
    model.fit(data)
    return model

In [ ]:
import pandas as pd
df = pd.read_csv("NSE_1001_TO_1050_start_to_15082020.csv")
df["Date"] = pd.to_datetime(df["Date"], format='mixed')
df["Close"] = df["Close"].astype(str).str.replace(",", "").astype(float)
df = df[["Date","Close"]]
model = train_prophet(df)
future = model.make_future_dataframe(periods=30)
forecast = model.predict(future)
print(forecast[["ds","yhat"]].tail())

In [ ]:
from prophet import Prophet
import pandas as pd

# Prepare data (Prophet needs specific column names)
prophet_df = df.reset_index()[['Date', 'Close']]
prophet_df.columns = ['ds', 'y']

# Train model
model = Prophet()
model.fit(prophet_df)

# Create next day dataframe
next_day_date = pd.Timestamp.now().normalize()
future = pd.DataFrame({'ds': [next_day_date]})

# Predict
forecast = model.predict(future)

# Same output format
prediction_df = pd.DataFrame({
    'Date': [next_day_date],
    'Predicted Value': [forecast['yhat'].iloc[0]]
})

display(prediction_df.set_index('Date'))

In [ ]:
current_date = pd.Timestamp.now().normalize()
future_week_dates = pd.date_range(current_date, periods=7)

future_week_data = pd.DataFrame(index=future_week_dates)
future_week_data['ds'] = future_week_dates # Add 'ds' column
future_week_data['year'] = future_week_data.index.year
future_week_data['month'] = future_week_data.index.month
future_week_data['day'] = future_week_data.index.day
future_week_data['dayofweek'] = future_week_data.index.dayofweek
future_week_data['dayofyear'] = future_week_data.index.dayofyear
future_week_data['weekofyear'] = future_week_data.index.isocalendar().week.astype(int)

# The features_for_prediction are not used by Prophet directly in this way,
# but if custom regressors were defined, they would be used.
# For standard Prophet prediction, only 'ds' is strictly required in the input dataframe.

weekly_predictions = model.predict(future_week_data)

weekly_prediction_df = pd.DataFrame({
    'Date': future_week_dates,
    'Predicted Value': weekly_predictions['yhat'] # Extract 'yhat' from the prediction result
})

display(weekly_prediction_df.set_index('Date'))

In [ ]:
%pip install statsmodels

In [ ]:
forecast.to_csv("predictions.csv")